# 🔋 Simulasi Extended Kalman Filter (EKF) Berbasis Model Baterai CSV

**Tujuan:** Mengimplementasikan algoritma *Extended Kalman Filter* (EKF) untuk estimasi State of Charge (SOC) baterai LiFePO4. Model matematika (OCV dan ECM 1RC) diimpor dari hasil ekstraksi yang disimpan dalam format `Model_Baterai_EKF.csv`. Simulasi ini memvalidasi algoritma pada dua jenis pengujian: Profil Beban Statis (Charge-Rest) dan Profil Beban Dinamis (Urban Load).

**Tuning Matriks EKF (Sesuai Bab 3):**
Berdasarkan hasil *Grid Search*, konfigurasi optimal matriks *Process Noise* ($Q$) dan *Measurement Noise* ($R$) yang menyesuaikan dengan resolusi instrumen ZKETECH EBC-A40L adalah:
$$Q_{\text{final}} = \begin{bmatrix} 10^{-5} & 0 \\ 0 & 10^{-4} \end{bmatrix}, \quad R_{\text{final}} = 10^{-4}\ \text{V}^2$$

**Dataset:**
1. `Model_Baterai_EKF.csv` (Berisi Kapasitas, Tabel OCV, dan Parameter R0, R1, C1).
2. `charge-rest 60m.csv` (Data simulasi beban statis / charging).
3. `Dynamic Profiling (Urban Load).csv` (Data simulasi beban dinamis).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
from scipy.interpolate import interp1d

# Konfigurasi Tampilan Plot
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (15, 6)
plt.rcParams['font.size'] = 12

---
## 1️⃣ IMPORT MODEL PARAMETER (DARI CUSTOM CSV)
Membaca file `Model_Baterai_EKF.csv` yang memiliki struktur multi-tabel. Data dipecah menjadi tiga bagian utama:
* **Battery Parameters:** Kapasitas nominal baterai dalam Coulomb ($Q_c$).
* **OCV Lookup Table:** Kurva relasi tegangan relaksasi terhadap SOC.
* **ECM Parameters:** Karakteristik dinamika sirkuit $R_0, R_1, C_1$.
Semua tabel diinterpolasi menjadi fungsi Python berkesinambungan.

In [ ]:
# =========================================================
# MEMUAT FILE MODEL BATERAI (MULTI-SECTION CSV)
# =========================================================
print("⏳ Memuat model dari Model_Baterai_EKF.csv...")

with open('Model_Baterai_EKF.csv', 'r') as f:
    text = f.read()

# 1. Ekstraksi Battery Parameters
param_str = text.split('=== BATTERY PARAMETERS ===')[1].split('=== OCV LOOKUP TABLE ===')[0].strip()
df_params = pd.read_csv(io.StringIO(param_str))
Q_Ah = float(df_params[df_params['Parameter'] == 'Q_Ah_Nominal']['Value'].values[0])
Q_Coulomb = float(df_params[df_params['Parameter'] == 'Q_As']['Value'].values[0])
print(f"✅ Kapasitas: {Q_Ah:.2f} Ah ({Q_Coulomb:.2f} Coulomb)")

# 2. Ekstraksi OCV Lookup Table
ocv_str = text.split('=== OCV LOOKUP TABLE ===')[1].split('=== ECM PARAMETERS (1-RC Thevenin) ===')[0].strip()
df_ocv = pd.read_csv(io.StringIO(ocv_str))
f_OCV_LUT = interp1d(
    df_ocv['SOC'], df_ocv['OCV_V'], 
    kind='linear', bounds_error=False, 
    fill_value=(df_ocv['OCV_V'].iloc[0], df_ocv['OCV_V'].iloc[-1])
)

# 3. Ekstraksi ECM Parameters
ecm_str = text.split('=== ECM PARAMETERS (1-RC Thevenin) ===')[1].strip()
df_ecm = pd.read_csv(io.StringIO(ecm_str))

# Bersihkan df_ecm: konversi SOC_percent ke numeric dan drop baris non-numeric
df_ecm['SOC_percent'] = pd.to_numeric(df_ecm['SOC_percent'], errors='coerce')
df_ecm = df_ecm.dropna()

# Perhatikan: Di CSV, SOC ECM biasanya dalam persentase, tapi untuk EKF kita butuh skala 0-1
soc_ecm_scale = df_ecm['SOC_percent'] 
# Jika di CSV SOC_percent bernilai 0, 10, 20... maka dibagi 100.
# Jika di CSV sudah 0.0, 0.1, 0.2... maka tidak perlu dibagi 100.
if soc_ecm_scale.max() > 2.0:
    soc_ecm_scale = soc_ecm_scale / 100.0

f_R0 = interp1d(soc_ecm_scale, df_ecm['R0_ohm'], fill_value='extrapolate')
f_R1 = interp1d(soc_ecm_scale, df_ecm['R1_ohm'], fill_value='extrapolate')
f_C1 = interp1d(soc_ecm_scale, df_ecm['C1_farad'], fill_value='extrapolate')

print("✅ Fungsi Interpolasi OCV dan ECM (R0, R1, C1) berhasil dibuat!")

---
## 2️⃣ IMPORT DATASET PENGUJIAN (URBAN LOAD)
Data pengujian profil dinamis diimpor dari file CSV hasil *logger* ZKETECH. Waktu cuplik (*sampling time* / $\Delta t$) diekstrak untuk menjamin diskretisasi matematis pada EKF berjalan sinkron dengan realita pengujian.

In [ ]:
# =========================================================
# MEMUAT DATASET BEBAN DINAMIS (URBAN LOAD)
# =========================================================
def load_zke_data(filepath):
    skip_rows = 0
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if "Time(S)" in line:
                skip_rows = i
                break
    df = pd.read_csv(filepath, skiprows=skip_rows)
    df.columns = [col.strip() for col in df.columns]
    return df

file_name = "Dynamic Profiling (Urban Load).csv"
df = load_zke_data(file_name)

# Kalkulasi Delta Time (dt)
df['dt'] = df['Time(S)'].diff().fillna(0)

# Potongan awal df untuk memastikan data aman
display(df[['Time(S)', 'Cur(A)', 'Vol(V)', 'dt']].head())
print(f"Total Baris Data: {len(df)}")

---
## 3️⃣ ALGORITMA EXTENDED KALMAN FILTER (EKF)

Model State-Space diskrit dieksekusi menggunakan algoritma Extended Kalman Filter (EKF). Terdapat beberapa penyesuaian khusus pada algoritma ini untuk memastikan stabilitas dan akurasi tinggi:
1. **Zero-Order Hold (ZOH):** Digunakan untuk diskretisasi eksak *time-update* komponen relaksasi $V_{c1}$.
2. **Numerical Jacobian:** Digunakan untuk menghindari kebutuhan turunan analitik polinomial pada relasi $OCV(SOC)$, mencegah divergensi di area *plateau* datar.
3. **Joseph Form Covariance Update:** Digunakan untuk memastikan stabilitas matriks (*Positive Semi-Definite*) pada komputasi jangka panjang.
4. **Parameter Tuning (Grid Search):** Berdasarkan hasil optimasi heuristik di Bab 3, matriks *process noise* ($Q$) dan *measurement noise* ($R$) ditetapkan pada nilai optimal:
   $$Q = \begin{bmatrix} 10^{-5} & 0 \\ 0 & 10^{-4} \end{bmatrix}, \quad R = 10^{-4}\ \text{V}^2$$

In [ ]:
# =========================================================
# KELAS EXTENDED KALMAN FILTER (EKF) - TUNED PARAMETERS
# =========================================================
class EKF_BMS:
    def __init__(self, Q_Coulomb, f_ocv):
        self.Q_c = Q_Coulomb  
        self.x = np.array([[1.0], [0.0]]) # State awal: [SOC=100%, Vc1=0V]
        self.P = np.array([[1e-2, 0], [0, 1e-2]])
        
        # =======================================================
        # TUNING Q DAN R BERDASARKAN HASIL GRID SEARCH BAB 3
        # =======================================================
        self.Q_noise = np.array([[1e-5, 0], 
                                 [0, 1e-4]]) 
        
        self.R_noise = np.array([[1e-4]]) 
        # =======================================================
        
        self.f_ocv = f_ocv
        self.I2 = np.eye(2) 
        
    def get_dOCV_dSOC(self, soc_val):
        """ Turunan Numerik Central Difference (Jacobian H) """
        delta = 0.001
        s_high = min(soc_val + delta, 1.0)
        s_low = max(soc_val - delta, 0.0)
        
        derivative = (float(self.f_ocv(s_high)) - float(self.f_ocv(s_low))) / (s_high - s_low)
        return max(float(derivative), 1e-6) # Proteksi agar Jacobian tidak Singular

    def step(self, I_meas, V_meas, dt):
        # Proteksi Batas Fisik SOC
        soc_prev = np.clip(float(self.x[0, 0]), 0.0, 1.0)
        vc1_prev = float(self.x[1, 0])
        
        # Ambil Parameter ECM sesuai SOC
        R0 = max(float(f_R0(soc_prev)), 1e-4)
        R1 = max(float(f_R1(soc_prev)), 1e-4)
        C1 = max(float(f_C1(soc_prev)), 1.0)
        tau = max(R1 * C1, 1e-6)
        
        # =====================================
        # TAHAP 1: PREDIKSI (A PRIORI)
        # =====================================
        soc_pred = soc_prev - (I_meas * dt / self.Q_c)
        soc_pred = np.clip(soc_pred, 0.0, 1.0)
        
        alpha = np.exp(-dt / tau) if dt > 0 else 1.0
        vc1_pred = (alpha * vc1_prev) + (R1 * (1.0 - alpha) * I_meas)
        
        self.x = np.array([[soc_pred], [vc1_pred]])
        A = np.array([[1.0, 0.0], [0.0, alpha]])
        self.P = A @ self.P @ A.T + self.Q_noise
        
        # =====================================
        # TAHAP 2: UPDATE (A POSTERIORI)
        # =====================================
        OCV_pred = float(self.f_ocv(soc_pred))
        dOCV_dSOC = self.get_dOCV_dSOC(soc_pred)
        
        V_pred = OCV_pred - vc1_pred - (I_meas * R0)
        H = np.array([[dOCV_dSOC, -1.0]])
        
        # Kalman Gain
        S = H @ self.P @ H.T + self.R_noise
        K = self.P @ H.T @ np.linalg.inv(S)
        
        error = V_meas - V_pred
        
        # Koreksi State
        self.x = self.x + (K * error)
        self.x[0, 0] = np.clip(self.x[0, 0], 0.0, 1.0)
        
        # Update Kovariansi P (Joseph Form)
        I_KH = self.I2 - K @ H
        self.P = I_KH @ self.P @ I_KH.T + K @ self.R_noise @ K.T
        
        return float(self.x[0, 0]), V_pred

---
## 4️⃣ EKSEKUSI SIMULASI DAN VALIDASI
Iterasi algoritma EKF dijalankan secara berurutan (*real-time simulation*) menggunakan profil beban dinamis. Sebagai referensi kalibrasi (*baseline*), perhitungan *Coulomb Counting* turut dikalkulasikan secara *Open-Loop*.

In [ ]:
# =========================================================
# SIMULASI COULOMB COUNTING & EKF
# =========================================================
print("⏳ Menjalankan Simulasi EKF...")

# Coulomb Counting Terpisah
df['Coulomb_Discharged'] = (df['Cur(A)'] * df['dt']).cumsum()
df['SOC_CC'] = 1.0 - (df['Coulomb_Discharged'] / Q_Coulomb)

# Eksekusi EKF
ekf = EKF_BMS(Q_Coulomb=Q_Coulomb, f_ocv=f_OCV_LUT)
soc_ekf_history = []
v_pred_history = []

for i in range(len(df)):
    soc_est, v_pred = ekf.step(df['Cur(A)'].iloc[i], df['Vol(V)'].iloc[i], df['dt'].iloc[i])
    soc_ekf_history.append(soc_est)
    v_pred_history.append(v_pred)

df['SOC_EKF'] = soc_ekf_history
df['V_Pred'] = v_pred_history

print("✅ Simulasi Selesai!")

# =========================================================
# VISUALISASI HASIL VALIDASI
# =========================================================
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(15, 12))
t_hrs = df['Time(S)'] / 3600.0

# 1. Plot Validasi Tegangan Model vs Observasi
ax1.plot(t_hrs, df['Vol(V)'], label='Tegangan Aktual Observasi (V)', color='red', alpha=0.5, linewidth=4)
ax1.plot(t_hrs, df['V_Pred'], label='Tegangan Estimasi EKF (V)', color='black', linestyle='--', linewidth=1.5)
ax1.set_ylabel('Voltage (V)', fontweight='bold')
ax1.set_title('Validasi Tegangan Terminal pada Profil Beban Dinamis (Urban Load)', fontweight='bold', fontsize=14)
ax1.legend(); ax1.grid(True, linestyle=':')

# 2. Plot Validasi SOC CC vs EKF
ax2.plot(t_hrs, df['SOC_CC'] * 100, label='Open-Loop: Coulomb Counting (%)', color='tab:orange', linestyle='-.', linewidth=2.5)
ax2.plot(t_hrs, df['SOC_EKF'] * 100, label='Closed-Loop: Extended Kalman Filter (%)', color='tab:green', linewidth=3)
ax2.fill_between(t_hrs, df['SOC_CC'] * 100, df['SOC_EKF'] * 100, color='red', alpha=0.1, label='Koreksi EKF (Kalman Adjustment)')

ax2.set_xlabel('Waktu Eksperimen (Jam)', fontweight='bold')
ax2.set_ylabel('State of Charge / SOC (%)', fontweight='bold')
ax2.set_title('Komparasi Performa Estimasi SOC: Coulomb Counting vs EKF', fontweight='bold', fontsize=14)
ax2.legend(); ax2.grid(True, linestyle=':')

plt.tight_layout()
plt.show()

# Print Metrik RMSE Tegangan
rmse_v = np.sqrt(np.mean((df['Vol(V)'] - df['V_Pred'])**2))
print("="*50)
print(f"📊 HASIL OPTIMASI EKF (Sesuai Grid Search Bab 3)")
print("="*50)
print(f"RMSE Validasi Tegangan EKF: {rmse_v*1000:.2f} mV")
print("="*50)